# Preference Learning: Training for Shorter Responses

This notebook demonstrates pairwise preference learning using Tinker to train a model that generates shorter, more concise responses.

**Goal**: Train the model to prefer shorter responses using pairwise comparisons.

**Method**: For each prompt, generate multiple responses and reward shorter ones that maintain quality.

In [ ]:
import os
os.environ['TINKER_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

## How Pairwise Preference RL Works

1. **Generate responses**: For each prompt, generate `group_size` responses
2. **Compare pairs**: Compare responses pairwise based on length
3. **Compute rewards**: Shorter responses get higher rewards
4. **Train**: Update model to favor shorter responses

### Reward Structure
- `win_minus_loss`: Score based on pairwise comparisons (-1 to 1)
- `format`: Whether response ends properly with EOS token
- Combined reward penalizes long, incomplete responses

In [ ]:
# Training configuration
config = {
    'model_name': 'Qwen/Qwen3-0.6B-Instruct',  # Small instruct model
    'group_size': 4,           # Responses per prompt for comparison
    'groups_per_batch': 50,    # Prompts per batch
    'learning_rate': 1e-4,
    'lora_rank': 32,
}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## Run Training

In [ ]:
!python -m tinker_cookbook.recipes.preference.shorter.train \
    model_name="Qwen/Qwen3-0.6B-Instruct" \
    group_size=4 \
    groups_per_batch=50 \
    learning_rate=1e-4

## Understanding the Output

### Trajectory Examples
```
trajectory idx=0, reward=-2
  win_minus_loss: -1.0  (longest response)
  format: False         (didn't end properly)

trajectory idx=3, reward=1
  win_minus_loss: 1.0   (shortest response)
  format: True          (ended properly)
```

### Key Metrics
| Metric | Description |
|--------|-------------|
| `env/all/reward/total` | Average reward (higher = shorter) |
| `env/all/format` | % of responses with proper EOS |
| `env/all/win_minus_loss` | Pairwise comparison score |
| `env/all/ac_tokens_per_turn` | Avg tokens per response (should decrease) |
| `kl_pre_post_v1` | KL divergence before/after update |

## Expected Behavior

As training progresses:
1. `ac_tokens_per_turn` decreases (shorter responses)
2. `format` increases (more proper endings)
3. `reward/total` increases (better overall)

Example progression:
```
Step 0:  tokens=60, format=47%, reward=-0.53
Step 10: tokens=45, format=80%, reward=0.20
Step 20: tokens=35, format=95%, reward=0.60
Step 32: tokens=30, format=98%, reward=0.85
```

## Analyze Results

In [ ]:
import json
import glob

# Find the latest metrics file
metrics_files = glob.glob('/tmp/tinker-examples/shorter/*/metrics.jsonl')
if metrics_files:
    latest = max(metrics_files)
    print(f"Reading: {latest}\n")
    
    tokens_history = []
    reward_history = []
    
    with open(latest) as f:
        for line in f:
            data = json.loads(line)
            if 'env/all/ac_tokens_per_turn' in data:
                tokens_history.append(data['env/all/ac_tokens_per_turn'])
                reward_history.append(data.get('env/all/reward/total', 0))
    
    if tokens_history:
        print(f"Starting tokens/response: {tokens_history[0]:.1f}")
        print(f"Current tokens/response: {tokens_history[-1]:.1f}")
        print(f"Reduction: {tokens_history[0] - tokens_history[-1]:.1f} tokens")
        print(f"\nReward: {reward_history[0]:.3f} -> {reward_history[-1]:.3f}")